Reference

https://www.kaggle.com/code/datasciencegrad/s6e2-detailed-eda-insights-w-blend/notebook

## Import Libraries and Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency

print("Libraries loaded successfully.")

In [ ]:
# Modern violet gradient color palette
COLORS = {
    'primary': '#667eea',
    'secondary': '#764ba2',
    'accent': '#f093fb',
    'highlight': '#4facfe',
    'dark': '#2d3561',
    'light': '#e0c3fc'
}

GRADIENT = ['#667eea', '#764ba2', '#a855f7', '#f093fb', '#fbc2eb']

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette(GRADIENT)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10

## Load Data

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
sample = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv')
original = pd.read_csv('/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f'Train Shape: {train.shape}')
print(f'Test Shape: {test.shape}')
print(f'Original Shape: {original.shape}')

## First Look at Data

In [ ]:
print("="*80)
print("TRAIN DATA - FIRST 5 ROWS")
print("="*80)
display(train.head())

print("\n" + "="*80)
print("TEST DATA - FIRST 5 ROWS")
print("="*80)
display(test.head())

print("\n" + "="*80)
print("ORIGINAL DATA - FIRST 5 ROWS")
print("="*80)
display(original.head())

## Data Types and Memory Usage

In [ ]:
print("="*80)
print("TRAIN DATA INFO")
print("="*80)
train.info()

print("\n" + "="*80)
print("ORIGINAL DATA INFO")
print("="*80)
original.info()

## Feature Identification

In [ ]:
TARGET = 'Churn'
ID_COL = 'id'

train_cols = [col for col in train.columns if col not in [ID_COL, TARGET]]
test_cols = [col for col in test.columns if col != ID_COL]

print(f"Target Variable: {TARGET}")
print(f"Number of Features: {len(train_cols)}")
print(f"\nFeature List:")
for i, col in enumerate(train_cols, 1):
    print(f"  {i:2d}. {col}")

# Identify categorical and numerical features
CATS = train[train_cols].select_dtypes(include=['object']).columns.tolist()
NUMS = train[train_cols].select_dtypes(exclude=['object']).columns.tolist()

print(f"\n{'='*80}")
print(f"Categorical Features ({len(CATS)}): {CATS}")
print(f"Numerical Features ({len(NUMS)}): {NUMS}")

## 1. Basic Data Quality Checks

#### Missing Values Analysis

In [ ]:
def check_missing_values(df, name):
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_df = pd.DataFrame({
        'Column': missing.index,
        'Missing_Count': missing.values,
        'Missing_Percentage': missing_pct.values
    })
    missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
    
    print(f"\n{'='*80}")
    print(f"{name} - MISSING VALUES ANALYSIS")
    print(f"{'='*80}")
    
    if len(missing_df) == 0:
        print("✓ No missing values found in the dataset.")
    else:
        display(missing_df)
    
    return missing_df

train_missing = check_missing_values(train, "TRAIN")
test_missing = check_missing_values(test, "TEST")
orig_missing = check_missing_values(original, "ORIGINAL")

#### Duplicate Rows Check

In [ ]:
print("="*80)
print("DUPLICATE ROWS CHECK")
print("="*80)

train_duplicates = train.duplicated().sum()
test_duplicates = test.duplicated().sum()
orig_duplicates = original.duplicated().sum()

print(f"Train Duplicates: {train_duplicates:,} ({train_duplicates/len(train)*100:.2f}%)")
print(f"Test Duplicates: {test_duplicates:,} ({test_duplicates/len(test)*100:.2f}%)")
print(f"Original Duplicates: {orig_duplicates:,} ({orig_duplicates/len(original)*100:.2f}%)")

### Unique Values Count

In [ ]:
print("="*80)
print("UNIQUE VALUES COUNT - ALL FEATURES")
print("="*80)

unique_counts = pd.DataFrame({
    'Feature': train_cols,
    'Train_Unique': [train[col].nunique() for col in train_cols],
    'Test_Unique': [test[col].nunique() for col in train_cols],
    'Original_Unique': [original[col].nunique() if col in original.columns else 0 for col in train_cols],
    'Train_Sample_Size': len(train),
    'Cardinality_Ratio': [train[col].nunique()/len(train)*100 for col in train_cols]
})

unique_counts = unique_counts.sort_values('Train_Unique', ascending=False)
display(unique_counts)

#### Statistical Summary - Numerical Features

In [ ]:
print("="*80)
print("STATISTICAL SUMMARY - NUMERICAL FEATURES (TRAIN)")
print("="*80)
display(train[NUMS].describe().T.style.background_gradient(cmap='viridis'))

print("\n" + "="*80)
print("STATISTICAL SUMMARY - NUMERICAL FEATURES (ORIGINAL)")
print("="*80)
orig_nums = [col for col in NUMS if col in original.columns]
display(original[orig_nums].describe().T.style.background_gradient(cmap='viridis'))

## 2. Target Variable Analysis

### Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train Target Distribution
target_counts_train = train[TARGET].value_counts().sort_index()
colors_train = [COLORS['primary'], COLORS['accent']]

axes[0].bar(target_counts_train.index, target_counts_train.values, color=colors_train, alpha=0.8, edgecolor='white', linewidth=2)
axes[0].set_title('Target Distribution - Train Dataset', fontsize=14, fontweight='bold', color=COLORS['dark'])
axes[0].set_xlabel(TARGET, fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].grid(axis='y', alpha=0.3)

for i, (label, count) in enumerate(target_counts_train.items()):
    pct = count / len(train) * 100
    axes[0].text(i, count + len(train)*0.01, f'{count:,}\n({pct:.1f}%)', 
                ha='center', va='bottom', fontsize=11, fontweight='bold')

# Original Target Distribution
if TARGET in original.columns:
    target_counts_orig = original[TARGET].value_counts().sort_index()
    axes[1].bar(target_counts_orig.index, target_counts_orig.values, color=colors_train, alpha=0.8, edgecolor='white', linewidth=2)
    axes[1].set_title('Target Distribution - Original Dataset', fontsize=14, fontweight='bold', color=COLORS['dark'])
    axes[1].set_xlabel(TARGET, fontsize=12)
    axes[1].set_ylabel('Count', fontsize=12)
    axes[1].grid(axis='y', alpha=0.3)
    
    for i, (label, count) in enumerate(target_counts_orig.items()):
        pct = count / len(original) * 100
        axes[1].text(i, count + len(original)*0.01, f'{count:,}\n({pct:.1f}%)', 
                    ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# Print class balance metrics
print("="*80)
print("CLASS BALANCE ANALYSIS")
print("="*80)
print(f"\nTrain Dataset:")
for label, count in target_counts_train.items():
    print(f"  {label}: {count:,} ({count/len(train)*100:.2f}%)")

imbalance_ratio = target_counts_train.max() / target_counts_train.min()
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}:1")

if imbalance_ratio > 1.5:
    print("⚠ Moderate class imbalance detected. Consider stratified CV and balanced metrics.")
elif imbalance_ratio > 2.0:
    print("⚠⚠ Significant class imbalance detected. Consider SMOTE or class weights.")
else:
    print("✓ Classes are relatively balanced.")

## 3. Univariate Analysis - Numerical Features

### Distribution of Numerical Features

In [ ]:
n_cols = 3
n_rows = (len(NUMS) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(NUMS):
    axes[i].hist(train[col], bins=50, color=COLORS['primary'], alpha=0.7, edgecolor='white')
    axes[i].set_title(f'{col}', fontsize=12, fontweight='bold', color=COLORS['dark'])
    axes[i].set_xlabel('Value', fontsize=10)
    axes[i].set_ylabel('Frequency', fontsize=10)
    axes[i].grid(axis='y', alpha=0.3)
    
    mean_val = train[col].mean()
    median_val = train[col].median()
    axes[i].axvline(mean_val, color=COLORS['secondary'], linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
    axes[i].axvline(median_val, color=COLORS['accent'], linestyle='--', linewidth=2, label=f'Median: {median_val:.2f}')
    axes[i].legend(fontsize=8)

for i in range(len(NUMS), len(axes)):
    axes[i].axis('off')

plt.suptitle('Distribution of Numerical Features - Train Dataset', fontsize=16, fontweight='bold', color=COLORS['dark'], y=1.002)
plt.tight_layout()
plt.show()

### Box Plots for Numerical Features (Outlier Detection)

In [ ]:
n_cols = 3
n_rows = (len(NUMS) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(NUMS):
    bp = axes[i].boxplot(train[col], vert=True, patch_artist=True, 
                         boxprops=dict(facecolor=COLORS['primary'], alpha=0.7),
                         medianprops=dict(color=COLORS['secondary'], linewidth=2),
                         whiskerprops=dict(color=COLORS['dark']),
                         capprops=dict(color=COLORS['dark']),
                         flierprops=dict(marker='o', markerfacecolor=COLORS['accent'], markersize=4, alpha=0.5))
    
    axes[i].set_title(f'{col}', fontsize=12, fontweight='bold', color=COLORS['dark'])
    axes[i].set_ylabel('Value', fontsize=10)
    axes[i].grid(axis='y', alpha=0.3)
    
    q1 = train[col].quantile(0.25)
    q3 = train[col].quantile(0.75)
    iqr = q3 - q1
    outliers = ((train[col] < (q1 - 1.5 * iqr)) | (train[col] > (q3 + 1.5 * iqr))).sum()
    outlier_pct = outliers / len(train) * 100
    
    axes[i].text(0.5, 0.95, f'Outliers: {outliers} ({outlier_pct:.1f}%)', 
                transform=axes[i].transAxes, ha='center', va='top',
                bbox=dict(boxstyle='round', facecolor=COLORS['light'], alpha=0.8), fontsize=9)

for i in range(len(NUMS), len(axes)):
    axes[i].axis('off')

plt.suptitle('Box Plots - Outlier Detection (Train Dataset)', fontsize=16, fontweight='bold', color=COLORS['dark'], y=1.002)
plt.tight_layout()
plt.show()

### Skewness and Kurtosis Analysis

In [ ]:
skew_kurt_df = pd.DataFrame({
    'Feature': NUMS,
    'Skewness': [train[col].skew() for col in NUMS],
    'Kurtosis': [train[col].kurtosis() for col in NUMS],
    'Mean': [train[col].mean() for col in NUMS],
    'Std': [train[col].std() for col in NUMS]
})

skew_kurt_df['Skew_Type'] = skew_kurt_df['Skewness'].apply(
    lambda x: 'Highly Right' if x > 1 else ('Right' if x > 0.5 else ('Symmetric' if abs(x) <= 0.5 else ('Left' if x < -0.5 else 'Highly Left')))
)

skew_kurt_df = skew_kurt_df.sort_values('Skewness', key=abs, ascending=False)

print("="*80)
print("SKEWNESS AND KURTOSIS ANALYSIS")
print("="*80)
display(skew_kurt_df.style.background_gradient(subset=['Skewness', 'Kurtosis'], cmap='coolwarm'))

print("\nInterpretation:")
print("  • Skewness > 1 or < -1: Highly skewed (consider log transform)")
print("  • Skewness between -0.5 and 0.5: Approximately symmetric")
print("  • Kurtosis > 3: Heavy tails (more outliers)")
print("  • Kurtosis < 3: Light tails (fewer outliers)")

## 4. Univariate Analysis - Categorical Features

### Categorical Features Distribution

In [ ]:
if len(CATS) > 0:
    n_cols = 2
    n_rows = (len(CATS) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()
    
    for i, col in enumerate(CATS):
        cat_counts = train[col].value_counts().sort_index()
        
        axes[i].bar(range(len(cat_counts)), cat_counts.values, color=GRADIENT[:len(cat_counts)], 
                   alpha=0.8, edgecolor='white', linewidth=2)
        axes[i].set_xticks(range(len(cat_counts)))
        axes[i].set_xticklabels(cat_counts.index, rotation=45, ha='right')
        axes[i].set_title(f'{col}', fontsize=12, fontweight='bold', color=COLORS['dark'])
        axes[i].set_ylabel('Count', fontsize=10)
        axes[i].grid(axis='y', alpha=0.3)
        
        for j, (label, count) in enumerate(cat_counts.items()):
            pct = count / len(train) * 100
            axes[i].text(j, count + len(train)*0.01, f'{count:,}\n({pct:.1f}%)', 
                        ha='center', va='bottom', fontsize=9)
    
    for i in range(len(CATS), len(axes)):
        axes[i].axis('off')
    
    plt.suptitle('Distribution of Categorical Features - Train Dataset', fontsize=16, fontweight='bold', color=COLORS['dark'], y=1.002)
    plt.tight_layout()
    plt.show()
else:
    print("No categorical features found.")

## 5. Bivariate Analysis - Features vs Target

In [ ]:
n_cols = 3
n_rows = (len(NUMS) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(NUMS):
    for target_val in sorted(train[TARGET].unique()):
        data = train[train[TARGET] == target_val][col]
        axes[i].hist(data, bins=30, alpha=0.6, label=f'{TARGET}={target_val}', 
                    color=COLORS['primary'] if target_val == train[TARGET].unique()[0] else COLORS['accent'],
                    edgecolor='white')
    
    axes[i].set_title(f'{col} vs {TARGET}', fontsize=12, fontweight='bold', color=COLORS['dark'])
    axes[i].set_xlabel('Value', fontsize=10)
    axes[i].set_ylabel('Frequency', fontsize=10)
    axes[i].legend(fontsize=9)
    axes[i].grid(axis='y', alpha=0.3)

for i in range(len(NUMS), len(axes)):
    axes[i].axis('off')

plt.suptitle('Numerical Features Distribution by Target', fontsize=16, fontweight='bold', color=COLORS['dark'], y=1.002)
plt.tight_layout()
plt.show()

### Box Plots - Numerical Features vs Target

In [ ]:
n_cols = 3
n_rows = (len(NUMS) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(NUMS):
    train_boxplot = train[[col, TARGET]].copy()
    train_boxplot[TARGET] = train_boxplot[TARGET].astype(str)
    
    bp = axes[i].boxplot([train[train[TARGET] == val][col].values for val in sorted(train[TARGET].unique())],
                         labels=sorted(train[TARGET].unique()),
                         patch_artist=True,
                         boxprops=dict(facecolor=COLORS['primary'], alpha=0.7),
                         medianprops=dict(color=COLORS['secondary'], linewidth=2),
                         whiskerprops=dict(color=COLORS['dark']),
                         capprops=dict(color=COLORS['dark']),
                         flierprops=dict(marker='o', markerfacecolor=COLORS['accent'], markersize=4, alpha=0.5))
    
    axes[i].set_title(f'{col} vs {TARGET}', fontsize=12, fontweight='bold', color=COLORS['dark'])
    axes[i].set_xlabel(TARGET, fontsize=10)
    axes[i].set_ylabel('Value', fontsize=10)
    axes[i].grid(axis='y', alpha=0.3)

for i in range(len(NUMS), len(axes)):
    axes[i].axis('off')

plt.suptitle('Box Plots - Numerical Features by Target', fontsize=16, fontweight='bold', color=COLORS['dark'], y=1.002)
plt.tight_layout()
plt.show()

### Statistical Tests - Numerical Features vs Target

In [ ]:
print("="*80)
print("STATISTICAL SIGNIFICANCE TESTS (Numerical Features vs Target)")
print("="*80)
print("\nUsing Mann-Whitney U Test (non-parametric)")
print("H0: The distributions are the same")
print("H1: The distributions are different")
print(f"Significance Level: α = 0.05\n")

significance_results = []

for col in NUMS:
    groups = [train[train[TARGET] == val][col].values for val in sorted(train[TARGET].unique())]
    
    if len(groups) == 2:
        stat, p_value = stats.mannwhitneyu(groups[0], groups[1], alternative='two-sided')
        
        significance_results.append({
            'Feature': col,
            'Test_Statistic': stat,
            'P_Value': p_value,
            'Significant': 'Yes' if p_value < 0.05 else 'No',
            'Effect': 'Strong' if p_value < 0.01 else ('Moderate' if p_value < 0.05 else 'Weak')
        })

sig_df = pd.DataFrame(significance_results).sort_values('P_Value')
display(sig_df.style.background_gradient(subset=['P_Value'], cmap='RdYlGn_r'))

print(f"\nSignificant Features (p < 0.05): {sig_df[sig_df['Significant'] == 'Yes'].shape[0]}/{len(NUMS)}")

### Categorical Features vs Target (if any)

In [ ]:
if len(CATS) > 0:
    n_cols = 2
    n_rows = (len(CATS) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()
    
    for i, col in enumerate(CATS):
        ct = pd.crosstab(train[col], train[TARGET], normalize='index') * 100
        ct.plot(kind='bar', stacked=False, ax=axes[i], color=[COLORS['primary'], COLORS['accent']], 
               alpha=0.8, edgecolor='white', linewidth=2)
        axes[i].set_title(f'{col} vs {TARGET}', fontsize=12, fontweight='bold', color=COLORS['dark'])
        axes[i].set_xlabel(col, fontsize=10)
        axes[i].set_ylabel('Percentage (%)', fontsize=10)
        axes[i].legend(title=TARGET, fontsize=9)
        axes[i].grid(axis='y', alpha=0.3)
        axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')
    
    for i in range(len(CATS), len(axes)):
        axes[i].axis('off')
    
    plt.suptitle('Categorical Features vs Target (Normalized)', fontsize=16, fontweight='bold', color=COLORS['dark'], y=1.002)
    plt.tight_layout()
    plt.show()
    
    # Chi-Square Tests
    print("\n" + "="*80)
    print("CHI-SQUARE TESTS (Categorical Features vs Target)")
    print("="*80)
    print("H0: Features are independent")
    print("H1: Features are dependent\n")
    
    chi_results = []
    
    for col in CATS:
        ct = pd.crosstab(train[col], train[TARGET])
        chi2, p_value, dof, expected = chi2_contingency(ct)
        
        chi_results.append({
            'Feature': col,
            'Chi2_Statistic': chi2,
            'P_Value': p_value,
            'DOF': dof,
            'Significant': 'Yes' if p_value < 0.05 else 'No'
        })
    
    chi_df = pd.DataFrame(chi_results).sort_values('P_Value')
    display(chi_df.style.background_gradient(subset=['P_Value'], cmap='RdYlGn_r'))
else:
    print("No categorical features to analyze.")

## 6. Multivariate Analysis

In [ ]:
corr_data = train[NUMS + [TARGET]].copy()

# Convert target to numeric if it's categorical
if corr_data[TARGET].dtype == 'object':
    target_mapping = {val: idx for idx, val in enumerate(sorted(corr_data[TARGET].unique()))}
    corr_data[TARGET] = corr_data[TARGET].map(target_mapping)

corr_matrix = corr_data.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='coolwarm', vmin=-1, vmax=1, center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            ax=ax)

ax.set_title('Correlation Matrix - Numerical Features & Target', 
            fontsize=16, fontweight='bold', color=COLORS['dark'], pad=20)

plt.tight_layout()
plt.show()

# Identify high correlations
print("="*80)
print("HIGH CORRELATIONS (|r| > 0.7)")
print("="*80)

high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.7:
            high_corr.append({
                'Feature_1': corr_matrix.columns[i],
                'Feature_2': corr_matrix.columns[j],
                'Correlation': corr_matrix.iloc[i, j]
            })

if high_corr:
    high_corr_df = pd.DataFrame(high_corr).sort_values('Correlation', key=abs, ascending=False)
    display(high_corr_df)
    print("\n⚠ High multicollinearity detected. Consider feature selection or regularization.")
else:
    print("✓ No high correlations found (|r| > 0.7)")

### Feature Correlations with Target

In [ ]:
target_corr = corr_matrix[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))

colors = [COLORS['primary'] if x > 0 else COLORS['accent'] for x in target_corr.values]
bars = ax.barh(range(len(target_corr)), target_corr.values, color=colors, alpha=0.8, edgecolor='white', linewidth=2)

ax.set_yticks(range(len(target_corr)))
ax.set_yticklabels(target_corr.index, fontsize=10)
ax.set_xlabel('Correlation Coefficient', fontsize=12)
ax.set_title(f'Feature Correlations with {TARGET}', fontsize=14, fontweight='bold', color=COLORS['dark'])
ax.axvline(0, color=COLORS['dark'], linewidth=1, linestyle='--')
ax.grid(axis='x', alpha=0.3)

for i, (feat, val) in enumerate(target_corr.items()):
    ax.text(val + (0.02 if val > 0 else -0.02), i, f'{val:.3f}', 
           va='center', ha='left' if val > 0 else 'right', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("="*80)
print(f"TOP 5 POSITIVELY CORRELATED FEATURES WITH {TARGET}")
print("="*80)
print(target_corr.head())

print("\n" + "="*80)
print(f"TOP 5 NEGATIVELY CORRELATED FEATURES WITH {TARGET}")
print("="*80)
print(target_corr.tail())

## 7. Distribution Comparison - Train vs Test vs Original

### Distribution Alignment Check

In [ ]:
df_combined = pd.DataFrame()

for col in NUMS:
    temp_train = train[col].copy()
    temp_train.name = 'value'
    temp_train_df = temp_train.to_frame()
    temp_train_df['source'] = 'Train'
    temp_train_df['feature'] = col
    
    temp_test = test[col].copy()
    temp_test.name = 'value'
    temp_test_df = temp_test.to_frame()
    temp_test_df['source'] = 'Test'
    temp_test_df['feature'] = col
    
    if col in original.columns:
        temp_orig = original[col].copy()
        temp_orig.name = 'value'
        temp_orig_df = temp_orig.to_frame()
        temp_orig_df['source'] = 'Original'
        temp_orig_df['feature'] = col
        
        df_combined = pd.concat([df_combined, temp_train_df, temp_test_df, temp_orig_df], ignore_index=True)
    else:
        df_combined = pd.concat([df_combined, temp_train_df, temp_test_df], ignore_index=True)

n_cols = 3
n_rows = (len(NUMS) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(NUMS):
    data_subset = df_combined[df_combined['feature'] == col]
    
    for source in ['Train', 'Test', 'Original']:
        source_data = data_subset[data_subset['source'] == source]['value']
        if len(source_data) > 0:
            axes[i].hist(source_data, bins=30, alpha=0.5, label=source, density=True)
    
    axes[i].set_title(f'{col}', fontsize=12, fontweight='bold', color=COLORS['dark'])
    axes[i].set_xlabel('Value', fontsize=10)
    axes[i].set_ylabel('Density', fontsize=10)
    axes[i].legend(fontsize=9)
    axes[i].grid(axis='y', alpha=0.3)

for i in range(len(NUMS), len(axes)):
    axes[i].axis('off')

plt.suptitle('Feature Distributions: Train vs Test vs Original', fontsize=16, fontweight='bold', color=COLORS['dark'], y=1.002)
plt.tight_layout()
plt.show()

### KS Test - Distribution Similarity (Train vs Test)

In [ ]:
print("="*80)
print("KOLMOGOROV-SMIRNOV TEST (Train vs Test)")
print("="*80)
print("H0: Distributions are the same")
print("H1: Distributions are different")
print(f"Significance Level: α = 0.05\n")

ks_results = []

for col in NUMS:
    stat, p_value = stats.ks_2samp(train[col], test[col])
    
    ks_results.append({
        'Feature': col,
        'KS_Statistic': stat,
        'P_Value': p_value,
        'Same_Distribution': 'Yes' if p_value > 0.05 else 'No',
        'Interpretation': 'Aligned' if p_value > 0.05 else 'Different'
    })

ks_df = pd.DataFrame(ks_results).sort_values('P_Value', ascending=False)
display(ks_df.style.background_gradient(subset=['P_Value'], cmap='RdYlGn'))

aligned_count = ks_df[ks_df['Same_Distribution'] == 'Yes'].shape[0]
print(f"\n✓ Aligned Features (Train vs Test): {aligned_count}/{len(NUMS)}")

if aligned_count / len(NUMS) > 0.8:
    print("✓ Train and Test distributions are well-aligned. Local CV should be reliable.")
else:
    print("⚠ Significant distribution differences detected. Consider domain adaptation techniques.")

## 8. Key Insights & Recommendations

### Summary Statistics Comparison

In [ ]:
summary_comparison = pd.DataFrame()

for col in NUMS:
    summary_comparison = pd.concat([summary_comparison, pd.DataFrame({
        'Feature': [col],
        'Train_Mean': [train[col].mean()],
        'Test_Mean': [test[col].mean()],
        'Train_Std': [train[col].std()],
        'Test_Std': [test[col].std()],
        'Train_Median': [train[col].median()],
        'Test_Median': [test[col].median()],
        'Mean_Diff_%': [abs(train[col].mean() - test[col].mean()) / train[col].mean() * 100]
    })], ignore_index=True)

summary_comparison = summary_comparison.sort_values('Mean_Diff_%', ascending=False)

print("="*80)
print("SUMMARY STATISTICS COMPARISON (Train vs Test)")
print("="*80)
display(summary_comparison.style.background_gradient(subset=['Mean_Diff_%'], cmap='YlOrRd'))

print("\n⚠ Features with >5% mean difference may indicate distribution shift.")

### Final Insights and Recommendations

In [ ]:
print("="*80)
print("KEY INSIGHTS & MODELING RECOMMENDATIONS")
print("="*80)

# 1. Data Quality
print("\n1. DATA QUALITY")
print("   " + "─"*75)
if train_missing.shape[0] == 0:
    print("   ✓ No missing values detected")
else:
    print(f"   ⚠ Missing values found in {train_missing.shape[0]} features")

if train_duplicates == 0:
    print("   ✓ No duplicate rows")
else:
    print(f"   ⚠ {train_duplicates:,} duplicate rows found")

# 2. Target Distribution
print("\n2. TARGET DISTRIBUTION")
print("   " + "─"*75)
target_counts = train[TARGET].value_counts()
balance_ratio = target_counts.max() / target_counts.min()
print(f"   • Class Balance Ratio: {balance_ratio:.2f}:1")

if balance_ratio < 1.5:
    print("   ✓ Classes are balanced - standard metrics (Accuracy) are appropriate")
elif balance_ratio < 3:
    print("   ⚠ Moderate imbalance - use Stratified CV and monitor AUC/F1")
else:
    print("   ⚠⚠ Significant imbalance - consider SMOTE, class weights, or focal loss")

# 3. Feature Characteristics
print("\n3. FEATURE CHARACTERISTICS")
print("   " + "─"*75)
print(f"   • Total Features: {len(NUMS)} numerical, {len(CATS)} categorical")
print(f"   • Highly Skewed Features: {skew_kurt_df[abs(skew_kurt_df['Skewness']) > 1].shape[0]}")
print(f"   • Features with Outliers: {len([col for col in NUMS if ((train[col] < train[col].quantile(0.25) - 1.5*(train[col].quantile(0.75)-train[col].quantile(0.25))) | (train[col] > train[col].quantile(0.75) + 1.5*(train[col].quantile(0.75)-train[col].quantile(0.25)))).sum() > 0])}")

# 4. Feature Importance Signals
print("\n4. FEATURE IMPORTANCE SIGNALS")
print("   " + "─"*75)
significant_features = sig_df[sig_df['Significant'] == 'Yes'].shape[0]
print(f"   • Statistically Significant Features: {significant_features}/{len(NUMS)}")
print(f"   • Top 3 Correlated with Target:")
for i, (feat, corr) in enumerate(target_corr.head(3).items(), 1):
    print(f"     {i}. {feat}: {corr:.3f}")

# 5. Distribution Alignment
print("\n5. TRAIN-TEST DISTRIBUTION ALIGNMENT")
print("   " + "─"*75)
aligned = ks_df[ks_df['Same_Distribution'] == 'Yes'].shape[0]
print(f"   • Aligned Features: {aligned}/{len(NUMS)} ({aligned/len(NUMS)*100:.1f}%)")

if aligned / len(NUMS) > 0.8:
    print("   ✓ Strong alignment - Local CV is reliable")
else:
    print("   ⚠ Weak alignment - CV scores may not reflect LB performance")

# 6. Multicollinearity
print("\n6. MULTICOLLINEARITY")
print("   " + "─"*75)
if high_corr:
    print(f"   ⚠ {len(high_corr)} feature pairs with |r| > 0.7")
    print("   → Consider PCA, feature selection, or L2 regularization")
else:
    print("   ✓ No high multicollinearity detected")

# 7. Recommended Modeling Strategy
print("\n7. RECOMMENDED MODELING STRATEGY")
print("   " + "─"*75)
print("   • Cross-Validation: 5-fold Stratified K-Fold")
print("   • Evaluation Metric: AUC-ROC (as per competition)")
print("   • Models to Try: XGBoost, LightGBM, CatBoost (ensemble)")
print("   • Feature Engineering:")
print("     - Target encoding on high-cardinality features")
print("     - Interaction terms between top correlated features")
print("     - Polynomial features for non-linear relationships")
if len(high_corr) > 0:
    print("     - Feature selection to address multicollinearity")
if skew_kurt_df[abs(skew_kurt_df['Skewness']) > 1].shape[0] > 0:
    print("     - Log/Box-Cox transform for skewed features")
print("   • Original Dataset Strategy:")
print("     - Use for external target encoding (leakage-free)")
print("     - Statistical features (orig_mean, orig_count)")

print("\n" + "="*80)
print("EDA COMPLETE")
print("="*80)

## 9. Baseline model

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

X = train[train.columns[1:-1]]
y = train.Churn == 'Yes'
X_test = test[test.columns[1:]]

cb = CatBoostClassifier(
    n_estimators=8000, 
    learning_rate=0.03,
    eval_metric='AUC', 
    auto_class_weights='Balanced',
    random_state=42,
    early_stopping_rounds=100,
    task_type='GPU',
    cat_features=CATS,
    verbose=False
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))
test_pred = np.zeros(len(X_test))

for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    cb.fit(X_train, y_train, eval_set=[(X_val, y_val)])

    val_pred = cb.predict_proba(X_val)[:,1]
    oof_pred[val_index] = val_pred   
    test_pred += cb.predict_proba(X_test)[:,1] / skf.n_splits

    print(" Fold AUC:", roc_auc_score(y_val, val_pred))

print("\nFinal CV AUC:", roc_auc_score(y, oof_pred))

submission = pd.DataFrame({
    "id": test["id"],
    TARGET: test_pred
})

submission.to_csv("submission.csv", index=False)

## 10. SHAP analysis

In [ ]:
import shap
explainer = shap.TreeExplainer(cb)
shap_values = explainer(X_val)

In [ ]:
shap.summary_plot(shap_values, plot_type="bar")

In [ ]:
shap.plots.beeswarm(shap_values)

In [ ]:
for name in X.columns:
    shap.dependence_plot(name, shap_values.values, X_val)

In [ ]:
# True Positive
ind = np.argmax((val_pred > 0.99)&(y_val == 1))
shap.plots.waterfall(shap_values[ind], max_display=14)

In [ ]:
# False Positive
ind = np.argmax((val_pred > 0.99)&(y_val == 0))
shap.plots.waterfall(shap_values[ind], max_display=14)

In [ ]:
# True Negative
ind = np.argmax((val_pred < 0.01)&(y_val == 0))
shap.plots.waterfall(shap_values[ind], max_display=14)

In [ ]:
# False Negative
ind = np.argmax((val_pred < 0.01)&(y_val == 1))
shap.plots.waterfall(shap_values[ind], max_display=14)